In [ ]:
# | default_exp features.breakpoint_motif_genomewide

In [ ]:
# | export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()

In [ ]:
# | export


class BreakPointMotifGenomewideEvaluator(FeatureEvaluator):
    """Extracts 4-mer adjacent breakpoint motifs for genomewide regions."""

    name = "BreakPointMotifGenomewide"
    source_file = ".BreakPointMotif.parquet"
    tier = 3
    category = "motifs"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted
            cols = set(df.columns)

            if "Motif" in cols and "Frequency" in cols:
                for row in df.to_dict("records"):
                    m = str(row["Motif"])
                    if pd.notna(row["Frequency"]):
                        extracted[f"bpmotif_gw_{m}"] = float(row["Frequency"])

            return extracted
        except Exception as e:
            log.exception("extraction_failed", evaluator=self.name, error=str(e))
            return {}

In [ ]:
# | test
evaluator = BreakPointMotifGenomewideEvaluator()
df = pd.DataFrame([{"Motif": "ATCG", "Frequency": 0.1}])
res = evaluator.extract(df)
assert res["bpmotif_gw_ATCG"] == 0.1